# MoE Backbone Tests on BDD100K

Testing the new MoE architectures (WP-M0) on actual BDD100K data streams. We will visualize how router probabilities differ between different driving conditions.
Run from the repository root or `notebooks/`. Start Jupyter with the project dependencies:

```powershell
uv run --extra cpu --with pandas --with matplotlib --with seaborn --with tqdm --with jupyterlab jupyter lab
```

Use `--extra cu124` instead of `--extra cpu` for the repository's CUDA environment. Set `BDD_ROOT` (or edit `bdd_root`) to the BDD100K directory. Set `MOE_CHECKPOINT` to a compatible encoder/full-method checkpoint for pretrained experiments; without it the notebook explicitly uses random initialization and trains the full backbone. The default 2,000-image cap bounds RAM use. `training_epochs`, `training_batches`, and `activation_iterations` control runtime.


In [ ]:
import os
import sys
from pathlib import Path

repo_root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "cafl4ds/configs").is_dir()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Open this notebook from within the cafl4ds repository.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from hydra.utils import instantiate
from loguru import logger
from omegaconf import OmegaConf

from cafl4ds.models.vit import MoEViTEncoder

# BDD100K configuration
IMAGE_SIZE = 64  # Change this variable to control image resolution
PATCH_SIZE = 8  # Match pretrained weights

bdd_root = Path(os.environ.get("BDD_ROOT", r"D:\AI Proj\bdd100k_images_100k")).expanduser()
bdd_split = "train"
images_dir = bdd_root / "images" / "100k" / bdd_split
labels_file = bdd_root / "labels" / f"bdd100k_labels_images_{bdd_split}.json"

source_cfg = OmegaConf.create(
    {
        "_target_": "cafl4ds.data.attributes.BDD100KSource",
        "bdd_root": str(bdd_root),
        "split": bdd_split,
        "images_dir": str(images_dir),
        "labels_file": str(labels_file),
        "img_size": IMAGE_SIZE,  # small size for quick testing
        "max_images": 2000,  # set to None to use all images
        "min_canary_count": 32,
    }
)

stream_cfg = OmegaConf.create(
    {
        "_target_": "cafl4ds.data.regime.RegimeStream",
        "batch_size": 32,
        "support_per_canary": 8,
        "query_per_canary": 8,
        "max_train_per_regime": 128,
        "canary_seed": 0,
    }
)
# The source caches decoded images in RAM; increase the cap deliberately.
source_cfg.legacy_labels = not labels_file.is_dir()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Set MOE_CHECKPOINT or this variable to an optional compatible encoder/full-method state dict.
# None explicitly runs a random-start experiment; pretrained claims do not apply.
checkpoint_path = Path(os.environ["MOE_CHECKPOINT"]).expanduser() if os.environ.get("MOE_CHECKPOINT") else None
training_epochs = 20
training_batches = 150
activation_iterations = 300


def initialize_encoder(encoder):
    if checkpoint_path is None:
        logger.warning("Random initialization: no MOE_CHECKPOINT supplied; the whole backbone is trainable.")
        return
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    state = {k.removeprefix("encoder."): v for k, v in checkpoint.items() if k.startswith("encoder.")}
    if not state:
        state = dict(checkpoint)  # Also accept an encoder-only state dict.
    if "pos_embed" in state and state["pos_embed"].shape != encoder.pos_embed.shape:
        import torch.nn.functional as F

        positions = state["pos_embed"]
        grid = int((positions.shape[1] - 1) ** 0.5)
        spatial = positions[:, 1:].reshape(1, grid, grid, -1).permute(0, 3, 1, 2)
        spatial = F.interpolate(
            spatial, size=(encoder.grid_size, encoder.grid_size), mode="bicubic", align_corners=False
        )
        state["pos_embed"] = torch.cat([positions[:, :1], spatial.flatten(2).transpose(1, 2)], dim=1)
    missing, unexpected = encoder.load_state_dict(state, strict=False)
    # A dense checkpoint may omit MoE MLPs, but must cover every shared backbone parameter.
    moe_prefixes = tuple(f"blocks.{i}.mlp." for i in encoder.moe_blocks)
    missing_shared = [key for key in missing if not key.startswith(moe_prefixes)]
    if missing_shared:
        raise ValueError(f"Checkpoint is missing shared backbone weights: {missing_shared}")
    logger.info("Loaded checkpoint {}; missing: {}; unexpected: {}", checkpoint_path, missing, unexpected)
    for name, param in encoder.named_parameters():
        param.requires_grad_(name.startswith(tuple(f"blocks.{i}." for i in encoder.moe_blocks)))

In [ ]:
source = instantiate(source_cfg)
stream = instantiate(stream_cfg, source=source, seed=13)
if not len(stream):
    raise ValueError("No training batches remain; increase max_images or reduce probe reservations.")
logger.info("Loaded {} batches; conditions: {}", len(stream), stream.era_names)

## Visualizing MoE Routing on BDD100K

Here we test the MoE Backbone routing logic on the BDD100K stream. We initialize our `MoETinyViTEncoder` and feed it a batch of images to check which experts process which tokens.

In [ ]:
torch.manual_seed(13)  # Paired initialization for the balancing ablation.
number_of_experts = 4

# --- Change this variable to test different Regimes ---
# e.g. 'daytime', 'night', 'dawn/dusk'
target_regime_name = "daytime"
target_regime_names = [target_regime_name]


# Initialize the MoE Encoder
encoder = MoEViTEncoder(
    img_size=IMAGE_SIZE,
    patch_size=PATCH_SIZE,
    in_chans=3,
    embed_dim=64,
    depth=4,
    num_heads=2,
    mlp_ratio=2.0,
    num_experts=number_of_experts,
    top_k=1,
    moe_blocks=(2, 3),
    routing_level="token",
    balancing_mode="batch",
)

initialize_encoder(encoder)
encoder = encoder.to(device)
encoder.eval()


if stream is not None and len(stream) > 0:
    target_era = 0
    if hasattr(stream, "regime_names") and hasattr(stream, "_regime_order"):
        try:
            # stream.regime_names is a dict[int, str]
            reg_id = next(k for k, v in stream.regime_names.items() if target_regime_name in v)
            target_era = stream._regime_order.index(reg_id)
        except (ValueError, StopIteration):
            logger.warning(f"Regime '{target_regime_name}' not found. Using first batch.")

    batch = next((b for b in stream if b.era == target_era), None)
    if batch is None:
        batch = next(iter(stream))
    device = next(encoder.parameters()).device
    imgs = batch.images.to(device)
    logger.info(f"Processing batch for '{target_regime_name}' with {imgs.shape[0]} images.")
else:
    # Fallback to dummy data if dataset is not found
    raise ValueError("Stream failed to load! Please check the BDD100K data path and memory usage.")

# Forward pass to get routing probabilities
with torch.no_grad():
    embeds, aux_loss = encoder.embed_with_aux(imgs)
probs = encoder.routing(imgs)
print(f"Auxiliary balancing loss: {aux_loss:.4f}")

### Routing Probabilities Visualization

We visualize the routing probabilities for each MoE block. A heatmap illustrates how tokens (y-axis) are distributed among the 4 experts (x-axis).

In [ ]:
probs_block2 = probs[encoder.moe_blocks[0]].detach().cpu().numpy()  # [B, T, E]
probs_block3 = probs[encoder.moe_blocks[1]].detach().cpu().numpy()  # [B, T, E]

sample_idx = 0  # Visualize the first image in the batch

# Calculate number of tokens and adjust annotations dynamically
num_tokens = probs_block2.shape[1]
show_annot = num_tokens <= 32
yticklabels = [f"{i}" for i in range(num_tokens)] if show_annot else False
fig_height = max(6, int(num_tokens * 0.3)) if show_annot else 8

fig, axes = plt.subplots(1, 2, figsize=(14, fig_height))

sns.heatmap(
    probs_block2[sample_idx],
    cmap="Blues",
    ax=axes[0],
    cbar_kws={"label": "Probability"},
    annot=show_annot,
    fmt=".2f",
    yticklabels=yticklabels,
)
axes[0].set_title("Encoder Layer 3 Routing (First Image)")
axes[0].set_xlabel("Expert Index")
axes[0].set_ylabel("Token Index (Spatial Patch)")

sns.heatmap(
    probs_block3[sample_idx],
    cmap="Oranges",
    ax=axes[1],
    cbar_kws={"label": "Probability"},
    annot=show_annot,
    fmt=".2f",
    yticklabels=yticklabels,
)
axes[1].set_title("Encoder Layer 4 Routing (First Image)")
axes[1].set_xlabel("Expert Index")
axes[1].set_ylabel("Token Index (Spatial Patch)")

plt.tight_layout()
plt.show()

### Spatial Routing Distribution (GradCAM-like)

We project the expert routing probabilities back to the spatial dimensions of the image. This allows us to see which parts of the image each expert specializes in processing.

In [ ]:
import math

import numpy as np
import torch.nn.functional as F

sample_idx = 0
img = imgs[sample_idx].cpu().detach().cpu().numpy().transpose(1, 2, 0)
# Normalize image for visualization
img = (img - img.min()) / (img.max() - img.min() + 1e-8)

# We visualize the spatial routing for the first MoE block
p_block = probs[encoder.moe_blocks[0]][sample_idx].detach().cpu()  # [T, E]
num_patches = (IMAGE_SIZE // PATCH_SIZE) ** 2
if p_block.shape[0] > num_patches:
    p_block = p_block[-num_patches:]  # Keep only spatial patches if class token is present

E = p_block.shape[1]
grid_size = int(np.sqrt(p_block.shape[0]))
img_size = imgs.shape[2:]

cols = math.ceil(E / 2)
fig, axes = plt.subplots(2, cols, figsize=(3 * cols, 6))
axes = axes.flatten()

vmax_val = p_block.max().item()  # Dynamic vmax for stronger heatmap

for e in range(E):
    expert_prob = p_block[:, e].view(1, 1, grid_size, grid_size)
    expert_prob_resized = F.interpolate(expert_prob, size=img_size, mode="bilinear", align_corners=False)
    expert_prob_resized = expert_prob_resized.squeeze().numpy()

    ax = axes[e]
    ax.imshow(img)
    im = ax.imshow(expert_prob_resized, cmap="jet", alpha=0.65, vmin=0, vmax=vmax_val)
    ax.set_title(f"Expert {e}")
    ax.axis("off")

# Hide any unused subplots
for e in range(E, len(axes)):
    axes[e].axis("off")

cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
fig.colorbar(im, cax=cbar_ax, label="Routing Probability")
plt.suptitle("Spatial Routing Distribution for First MoE Block")
plt.tight_layout(rect=[0, 0, 0.9, 1])
plt.show()

### Mean Expert Usage

Let's look at the overall expert usage across the entire batch.

In [ ]:
# Compute average expert load across the batch and all tokens
mean_usage_block2 = probs_block2.mean(axis=(0, 1))
mean_usage_block3 = probs_block3.mean(axis=(0, 1))

fig, ax = plt.subplots(figsize=(8, 5))
width = 0.35
x = range(number_of_experts)

ax.bar([i - width / 2 for i in x], mean_usage_block2, width, label="Block 2", color="steelblue")
ax.bar([i + width / 2 for i in x], mean_usage_block3, width, label="Block 3", color="darkorange")

ax.set_xlabel("Expert Index")
ax.set_ylabel("Average Routing Probability")
ax.set_title("Mean Expert Usage across Batch")
ax.set_xticks(x)
ax.set_xticklabels([f"{i}" for i in x])
ax.legend()
plt.show()

## Regime Routing Comparison

Here we systematically compare how different regimes (e.g. daytime vs night vs dawn/dusk) influence the expert routing distribution.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

regimes_to_test = ["daytime", "night", "dawn/dusk"]
expert_counts_per_regime = {}

if stream is not None:
    for regime_name in regimes_to_test:
        try:
            reg_id = next(k for k, v in stream.regime_names.items() if regime_name in v)
            target_era = stream._regime_order.index(reg_id)
            batch = next((b for b in stream if b.era == target_era), None)
            if batch is not None:
                device = next(encoder.parameters()).device
                imgs = batch.images.to(device)
                with torch.no_grad():
                    probs = encoder.routing(imgs)
                # probs is a list of tensors of shape [batch_size * seq_len, num_experts]
                agg_probs = []
                for prob in probs.values():
                    avg_prob = prob.mean(dim=(0, 1)).cpu().numpy()
                    agg_probs.append(avg_prob)
                expert_counts_per_regime[regime_name] = agg_probs
        except (ValueError, StopIteration):
            logger.warning(f"Regime containing '{regime_name}' not found in stream.")

if expert_counts_per_regime:
    num_blocks = len(next(iter(expert_counts_per_regime.values())))
    fig, axes = plt.subplots(1, num_blocks, figsize=(6 * num_blocks, 5))
    if num_blocks == 1:
        axes = [axes]

    for i in range(num_blocks):
        ax = axes[i]
        num_experts = len(next(iter(expert_counts_per_regime.values()))[i])
        x = np.arange(num_experts)
        width = 0.25
        multiplier = 0

        for regime_name, agg_probs in expert_counts_per_regime.items():
            offset = width * multiplier
            rects = ax.bar(x + offset, agg_probs[i], width, label=regime_name)
            multiplier += 1

        ax.set_title(f"Average Expert Probabilities (MoE Block {i})")
        ax.set_xlabel("Expert Index")
        ax.set_ylabel("Average Probability")
        ax.set_xticks(x + width * (len(expert_counts_per_regime) - 1) / 2)
        ax.set_xticklabels([f"E{j}" for j in range(num_experts)])
        ax.legend(loc="upper right")

    plt.tight_layout()
    plt.show()
else:
    print("No valid regimes found for comparison.")

## Training the MoE Encoder via SimSiam

We hook up our `MoETinyViTEncoder` into the `SimSiam` SSL framework and train it for 250 batches. This will allow the router weights to learn from the data, meaning experts will start to specialize semantically. After training, we can re-run the routing comparison to observe meaningful differences!

In [ ]:
import numpy as np
import torch.optim as optim
from tqdm.auto import tqdm

from cafl4ds.measurements import rankme
from cafl4ds.ssl.factory import build_simsiam

simsiam = build_simsiam(encoder, aux_weight=0.01).to(device)
optimizer = optim.AdamW((p for p in simsiam.parameters() if p.requires_grad), lr=1e-5, weight_decay=1e-4)
history = {"epoch": [], "loss": [], "rankme": []}
allowed_eras = {era for era, name in stream.era_names.items() if any(t in name for t in target_regime_names)}
if not allowed_eras:
    raise ValueError(f"No batches match {target_regime_names}; available: {stream.era_names}")


def training_batches_for_regimes():
    # Repeat eligible batches without materializing another image copy or exhausting a filtered iterator.
    while True:
        found = False
        for batch in stream:
            if batch.era in allowed_eras and len(batch.images) >= 2:
                found = True
                yield batch
        if not found:
            raise ValueError("The requested regimes have no batches of at least two images.")


train_iterator = training_batches_for_regimes()
for epoch in range(training_epochs):
    simsiam.train()
    epoch_losses, all_embeddings = [], []
    for step in tqdm(range(training_batches), desc=f"Epoch {epoch + 1}/{training_epochs}"):
        imgs = next(train_iterator).images.to(device)
        optimizer.zero_grad()
        loss = simsiam.training_step(imgs)
        loss.backward()
        optimizer.step()
        epoch_losses.append(loss.item())
        if step >= max(0, training_batches - 20):
            encoder.eval()
            with torch.no_grad():
                all_embeddings.append(encoder.embed(imgs).cpu())
            simsiam.train()
    if not all_embeddings:
        raise ValueError("training_batches must be positive.")
    rm = rankme(torch.cat(all_embeddings, dim=0))
    history["epoch"].append(epoch + 1)
    history["loss"].append(float(np.mean(epoch_losses)))
    history["rankme"].append(rm)
    logger.info("Epoch {} | Loss: {:.4f} | RankMe: {:.2f}", epoch + 1, history["loss"][-1], rm)
encoder.eval()

In [ ]:
# Plot training performance over epochs
fig, ax1 = plt.subplots(figsize=(8, 5))

color = "tab:red"
ax1.set_xlabel("Epoch")
ax1.set_ylabel("SimSiam Loss", color=color)
ax1.plot(history["epoch"], history["loss"], color=color, marker="o", label="Loss")
ax1.tick_params(axis="y", labelcolor=color)
ax1.set_xticks(history["epoch"])

ax2 = ax1.twinx()
color = "tab:blue"
ax2.set_ylabel("RankMe (Collapse Measure)", color=color)
ax2.plot(history["epoch"], history["rankme"], color=color, marker="s", label="RankMe")
ax2.tick_params(axis="y", labelcolor=color)

fig.tight_layout()
plt.title("Training Performance: Loss and Representation RankMe")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

regimes_to_test = ["daytime", "night", "dawn/dusk"]
expert_counts_per_regime = {}

if stream is not None:
    for regime_name in regimes_to_test:
        try:
            reg_id = next(k for k, v in stream.regime_names.items() if regime_name in v)
            target_era = stream._regime_order.index(reg_id)
            batch = next((b for b in stream if b.era == target_era), None)
            if batch is not None:
                device = next(encoder.parameters()).device
                imgs = batch.images.to(device)
                with torch.no_grad():
                    probs = encoder.routing(imgs)
                # probs is a list of tensors of shape [batch_size * seq_len, num_experts]
                agg_probs = []
                for prob in probs.values():
                    avg_prob = prob.mean(dim=(0, 1)).cpu().numpy()
                    agg_probs.append(avg_prob)
                expert_counts_per_regime[regime_name] = agg_probs
        except (ValueError, StopIteration):
            logger.warning(f"Regime containing '{regime_name}' not found in stream.")

if expert_counts_per_regime:
    num_blocks = len(next(iter(expert_counts_per_regime.values())))
    fig, axes = plt.subplots(1, num_blocks, figsize=(6 * num_blocks, 5))
    if num_blocks == 1:
        axes = [axes]

    for i in range(num_blocks):
        ax = axes[i]
        num_experts = len(next(iter(expert_counts_per_regime.values()))[i])
        x = np.arange(num_experts)
        width = 0.25
        multiplier = 0

        for regime_name, agg_probs in expert_counts_per_regime.items():
            offset = width * multiplier
            rects = ax.bar(x + offset, agg_probs[i], width, label=regime_name)
            multiplier += 1

        ax.set_title(f"Average Expert Probabilities (MoE Block {i})")
        ax.set_xlabel("Expert Index")
        ax.set_ylabel("Average Probability")
        ax.set_xticks(x + width * (len(expert_counts_per_regime) - 1) / 2)
        ax.set_xticklabels([f"E{j}" for j in range(num_experts)])
        ax.legend(loc="upper right")

    plt.tight_layout()
    plt.show()
else:
    print("No valid regimes found for comparison.")

### Routing Probabilities Visualization

We visualize the routing probabilities for each MoE block. A heatmap illustrates how tokens (y-axis) are distributed among the 4 experts (x-axis).

In [ ]:
probs_block2 = probs[encoder.moe_blocks[0]].detach().cpu().numpy()  # [B, T, E]
probs_block3 = probs[encoder.moe_blocks[1]].detach().cpu().numpy()  # [B, T, E]

sample_idx = 0  # Visualize the first image in the batch

# Calculate number of tokens and adjust annotations dynamically
num_tokens = probs_block2.shape[1]
show_annot = num_tokens <= 32
yticklabels = [f"{i}" for i in range(num_tokens)] if show_annot else False
fig_height = max(6, int(num_tokens * 0.3)) if show_annot else 8

fig, axes = plt.subplots(1, 2, figsize=(14, fig_height))

sns.heatmap(
    probs_block2[sample_idx],
    cmap="Blues",
    ax=axes[0],
    cbar_kws={"label": "Probability"},
    annot=show_annot,
    fmt=".2f",
    yticklabels=yticklabels,
)
axes[0].set_title("Encoder Layer 3 Routing (First Image)")
axes[0].set_xlabel("Expert Index")
axes[0].set_ylabel("Token Index (Spatial Patch)")

sns.heatmap(
    probs_block3[sample_idx],
    cmap="Oranges",
    ax=axes[1],
    cbar_kws={"label": "Probability"},
    annot=show_annot,
    fmt=".2f",
    yticklabels=yticklabels,
)
axes[1].set_title("Encoder Layer 4 Routing (First Image)")
axes[1].set_xlabel("Expert Index")
axes[1].set_ylabel("Token Index (Spatial Patch)")

plt.tight_layout()
plt.show()

### Spatial Routing Distribution (GradCAM-like)

We project the expert routing probabilities back to the spatial dimensions of the image. This allows us to see which parts of the image each expert specializes in processing.

In [ ]:
import math

import numpy as np
import torch.nn.functional as F

sample_idx = 0
img = imgs[sample_idx].cpu().detach().cpu().numpy().transpose(1, 2, 0)
# Normalize image for visualization
img = (img - img.min()) / (img.max() - img.min() + 1e-8)

# We visualize the spatial routing for the first MoE block
p_block = probs[encoder.moe_blocks[0]][sample_idx].detach().cpu()  # [T, E]
num_patches = (IMAGE_SIZE // PATCH_SIZE) ** 2
if p_block.shape[0] > num_patches:
    p_block = p_block[-num_patches:]  # Keep only spatial patches if class token is present

E = p_block.shape[1]
grid_size = int(np.sqrt(p_block.shape[0]))
img_size = imgs.shape[2:]

cols = math.ceil(E / 2)
fig, axes = plt.subplots(2, cols, figsize=(3 * cols, 6))
axes = axes.flatten()

vmax_val = p_block.max().item()  # Dynamic vmax for stronger heatmap

for e in range(E):
    expert_prob = p_block[:, e].view(1, 1, grid_size, grid_size)
    expert_prob_resized = F.interpolate(expert_prob, size=img_size, mode="bilinear", align_corners=False)
    expert_prob_resized = expert_prob_resized.squeeze().numpy()

    ax = axes[e]
    ax.imshow(img)
    im = ax.imshow(expert_prob_resized, cmap="jet", alpha=0.65, vmin=0, vmax=vmax_val)
    ax.set_title(f"Expert {e}")
    ax.axis("off")

# Hide any unused subplots
for e in range(E, len(axes)):
    axes[e].axis("off")

cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
fig.colorbar(im, cax=cbar_ax, label="Routing Probability")
plt.suptitle("Spatial Routing Distribution for First MoE Block")
plt.tight_layout(rect=[0, 0, 0.9, 1])
plt.show()